In [26]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# from linear_combination import generate_mixture_spectra
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Set a nice default style
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 12

In [27]:
df = pd.read_excel('Spectral_library_clean.xlsx')
df.ffill(axis=0, inplace=True)
df.bfill(axis=0, inplace=True)  # handles leading NaNs

wavelength = df['Wavelength']
abs_spectra = df.iloc[:, 1:].to_numpy().T

In [28]:
df_mix_spec = pd.read_excel("Spectral_library_with_mixtures.xlsx")
df_mix_spec.ffill(axis=0, inplace=True)
df_mix_spec.bfill(axis=0, inplace=True)

N_species = abs_spectra.shape[0]
N_mix = df_mix_spec.shape[1]

df_mix_weights = pd.read_excel("Spectral_library_mixture_weights.xlsx")

mixture_cols = df_mix_weights["mixture_name"].tolist()
mix_spectra = df_mix_spec[mixture_cols].to_numpy().T

In [29]:
species_cols = list(df_mix_weights.columns[1:])
mixture_cols = df_mix_weights["mixture_name"].tolist()
mix_spectra = df_mix_spec[mixture_cols].to_numpy().T

df_mixtures = pd.DataFrame(
    data=mix_spectra.T,
    columns=mixture_cols
)

wavelength_mix = df_mix_spec["Wavelength"].to_numpy()
df_mixtures.insert(0, "Wavelength", wavelength_mix)

In [30]:
X = df_mix_spec[mixture_cols].to_numpy().T
Y = df_mix_weights[species_cols].to_numpy()

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.9, random_state=0)

noise_level = 0.01

X_train_noisy = X_train + np.random.normal(0, noise_level, X_train.shape)
Y_train_noisy = Y_train + np.random.normal(0, noise_level, Y_train.shape)
X_test_noisy = X_test + np.random.normal(0, noise_level, X_test.shape)

In [31]:
pls = PLSRegression(n_components=N_species)

# Train the model
pls.fit(X_train_noisy, Y_train)

Y_pred_raw = pls.predict(X_test_noisy)

Y_pred = np.clip(Y_pred_raw, 0, None)
row_sums = Y_pred.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
Y_pred_clean = Y_pred / row_sums

mae = mean_absolute_error(Y_test, Y_pred_clean)
print(f"Overall Mean Absolute Error: {mae:.4f}")

for i in range(len(Y_test)):
    print(f"Test Mixture {i+1}:")
    print(f"  True : {np.round(Y_test[i], 3)}")
    print(f"  PLS  : {np.round(Y_pred_clean[i], 3)}")

Overall Mean Absolute Error: 0.0809
Test Mixture 1:
  True : [0.    0.    0.052 0.766 0.    0.182 0.    0.    0.   ]
  PLS  : [0.109 0.326 0.118 0.204 0.056 0.186 0.    0.    0.   ]
Test Mixture 2:
  True : [0.    0.    0.    0.676 0.    0.    0.    0.324 0.   ]
  PLS  : [0.198 0.156 0.208 0.051 0.    0.073 0.042 0.216 0.056]
Test Mixture 3:
  True : [0.295 0.    0.651 0.    0.054 0.    0.    0.    0.   ]
  PLS  : [0.175 0.129 0.363 0.221 0.096 0.    0.015 0.    0.   ]
Test Mixture 4:
  True : [0.    0.    0.627 0.    0.    0.148 0.    0.225 0.   ]
  PLS  : [0.    0.06  0.502 0.038 0.    0.13  0.057 0.198 0.015]
Test Mixture 5:
  True : [0.384 0.1   0.    0.    0.    0.    0.    0.516 0.   ]
  PLS  : [0.252 0.083 0.    0.133 0.    0.    0.031 0.459 0.041]
Test Mixture 6:
  True : [0.    0.    0.    0.    0.334 0.    0.576 0.09  0.   ]
  PLS  : [0.    0.    0.06  0.    0.323 0.    0.366 0.191 0.058]
Test Mixture 7:
  True : [0.    0.    0.151 0.791 0.    0.058 0.    0.    0.   ]
  PLS  

/Users/jochem1411/Library/CloudStorage/OneDrive-UvA/Year 1 - MSc AMEP/Semester 2/Machine Learning/.venv/lib/python3.9/site-packages/sklearn/cross_decomposition/_pls.py:534: RuntimeWarning: divide by zero encountered in matmul
  Ypred = X @ self.coef_.T + self.intercept_
/Users/jochem1411/Library/CloudStorage/OneDrive-UvA/Year 1 - MSc AMEP/Semester 2/Machine Learning/.venv/lib/python3.9/site-packages/sklearn/cross_decomposition/_pls.py:534: RuntimeWarning: overflow encountered in matmul
  Ypred = X @ self.coef_.T + self.intercept_
/Users/jochem1411/Library/CloudStorage/OneDrive-UvA/Year 1 - MSc AMEP/Semester 2/Machine Learning/.venv/lib/python3.9/site-packages/sklearn/cross_decomposition/_pls.py:534: RuntimeWarning: invalid value encountered in matmul
  Ypred = X @ self.coef_.T + self.intercept_


In [33]:
pred_all = np.argsort(Y_test, axis = 1)
actual_all = np.argsort(Y_pred_clean, axis = 1)
print(pred_all, actual_all)
# checks if whole list is in correct order

accuracy = np.mean(pred_all == actual_all) * 100
print(accuracy)
print(f'Random guess:{100/9}')

[[0 1 4 6 7 8 2 5 3]
 [0 1 2 4 5 6 8 7 3]
 [1 3 5 6 7 8 4 0 2]
 [0 1 3 4 6 8 5 7 2]
 [2 3 4 5 6 8 1 0 7]
 [0 1 2 3 5 8 7 4 6]
 [0 1 4 6 7 8 5 2 3]
 [0 1 2 4 6 8 7 5 3]
 [0 1 4 5 7 8 6 2 3]
 [0 1 2 3 4 6 7 8 5]
 [1 2 3 4 5 6 8 7 0]
 [0 2 3 4 6 8 5 7 1]
 [0 1 2 3 4 5 7 6 8]
 [1 2 3 5 6 8 0 7 4]
 [0 2 4 5 6 7 8 3 1]
 [0 1 3 5 7 8 6 4 2]
 [0 1 3 4 6 7 5 8 2]
 [0 1 3 4 6 7 8 2 5]
 [2 4 5 6 7 8 0 3 1]
 [0 2 4 5 6 8 1 3 7]
 [0 1 2 4 7 8 3 5 6]
 [0 1 3 4 6 7 5 8 2]
 [0 3 4 5 6 7 8 2 1]
 [2 3 5 6 7 8 1 0 4]
 [0 1 3 4 6 8 2 5 7]
 [2 4 5 6 7 8 3 1 0]
 [1 2 3 4 6 8 5 7 0]
 [0 1 3 4 5 6 8 7 2]
 [0 1 4 5 6 7 2 8 3]
 [0 1 2 3 4 6 7 5 8]
 [0 3 4 5 6 7 2 1 8]
 [0 1 3 4 5 6 2 7 8]
 [0 2 4 5 6 8 1 7 3]
 [1 2 4 5 6 7 8 0 3]
 [0 1 2 4 5 6 7 8 3]
 [0 1 2 3 4 8 7 6 5]
 [0 1 3 4 5 7 8 6 2]
 [0 3 4 6 7 8 2 5 1]
 [2 3 4 5 7 8 1 0 6]
 [2 3 4 5 6 7 8 1 0]
 [0 3 4 5 6 8 1 2 7]
 [1 2 3 4 5 7 8 6 0]
 [0 4 5 6 7 8 2 3 1]
 [2 3 5 6 7 8 1 4 0]
 [1 3 4 5 6 8 7 0 2]
 [0 2 3 5 6 7 8 4 1]
 [0 2 5 6 7 8 4 1 3]
 [1 3 4 5 6 7